---

**Project:** Extend [arXiv:2501.07951](https://arxiv.org/abs/2501.07951) — Monte Carlo-based Parameter Reconstruction of an Optical Quantum System  
**Goal:** Convert the MC simulation into a backpropagatable pipeline (PyTorch) and replace grid search with gradient-based optimization.

---

## Terminology

| Term | Meaning |
|------|---------|
| **Run** | One MC trial — see [Terminology Reference](#terminology--detailed-reference) below |
| **Simulation** | The full ensemble of $N$ runs — see below |
| **MC Distribution** | The histogram of linewidths from one simulation |

---

## Terminology — Detailed Reference

### One Run

A **run** is a single Monte Carlo trial. It takes a fixed set of parameters 
$(\gamma, \bar{n}, \sigma, \lambda)$ and produces **one extracted linewidth** $w_i$.

**Step-by-step:**

1. **Compute the noiseless PLE spectrum** — For a grid of laser frequencies $\omega$, compute the expected absorption probability using a Lorentzian lineshape centered at $\omega_0$ with HWHM $\gamma$, scaled by the mean photon number $\bar{n}$.

2. **Add noise** — Simulate the measurement: Poisson noise from finite photon counts (shot noise) + Gaussian readout noise with std $\sigma$.

3. **Fit a Lorentzian** — Fit the noisy spectrum to extract an estimate of the linewidth. This gives us $w_i$, our one measurement for this run.

Each run produces a scalar: $w_i$ (the extracted linewidth). Repeating a run with the same parameters gives a *different* $w_i$ each time because the noise is random. The distribution of $w_i$ across many runs is what matters.

> **Key insight:** A single run is worthless on its own. Its value comes from being one sample in the distribution.

---

### One Simulation

A **simulation** is the collection of many runs (typically $N = 2000$) all performed with the **same** underlying parameters $(\gamma, \bar{n}, \sigma, \lambda)$. The output is a **histogram of extracted linewidths**:

$$\{w_1, w_2, ..., w_N\}$$

This histogram represents the distribution of linewidths that the measurement would produce under those specific physical parameters, given the noise in the system.

**What you do with it:** Compare this histogram to the experimental (real) linewidth histogram using a $\chi^2$ test. The goal of the original paper is to find $(\gamma, \bar{n})$ that make the simulated histogram best match the real data — hence the grid search over these parameters.

---

### Analogy

Think of it like archery:

| Concept | Archery analogy |
|---------|----------------|
| **True parameters** $(\gamma, \bar{n})$ | The archer's skill (steadiness, aim) |
| **Run** | One arrow shot |
| **Simulation** | 2000 arrows shot under the same conditions |
| **Linewidth histogram** | The scatter pattern on the target |
| **$\chi^2$ comparison** | Checking if this scatter pattern matches the pattern from a known archer |

You can't judge the archer from one arrow. You need the full scatter pattern.

---

### Why This Matters for Differentiable MC

In the original paper, the simulation is used as a black box inside a grid search. To make it differentiable, we need to:

1. Replace the discrete histogram + $\chi^2$ with a smooth density + differentiable divergence
2. Make the noise sampling differentiable (reparameterization trick, STE, etc.)
3. Optimize parameters via gradient descent instead of scanning a grid

---

## Chapter 1: The Original MC Algorithm (from the paper)

The paper [arXiv:2501.07951](https://arxiv.org/abs/2501.07951) introduces a Monte Carlo method to reconstruct optical linewidth parameters from low-signal PLE spectroscopy data of NV centers in diamond.

### Algorithm Summary

For a fixed set of parameters $(\gamma_{\text{true}}, \bar{n}, \sigma)$:

1. **Sample the total photon count** — Draw the number of detected photons $n$ from a normal distribution $\mathcal{N}(\bar{n}, \sigma)$. The mean $\bar{n}$ and standard deviation $\sigma$ represent the expected signal strength and its fluctuation.

2. **Distribute photons across frequency** — Place $n$ detection events across a fixed frequency range (150 MHz) according to a **Cauchy distribution** (Lorentzian):

$$P(\omega, \gamma) = \frac{\gamma / \pi}{\omega^2 + \gamma^2}$$

   where $\gamma$ is the half-width at half-maximum (HWHM) — the physical linewidth we want to estimate.

3. **Add noise** — Sample the number of background noise events from a Poisson distribution with mean 2 (see Supplemental Material for details). These are distributed uniformly across the frequency range to simulate dark counts and background.

4. **Fit the spectrum** — The resulting noisy spectrum (signal + noise binned in frequency) is fitted with a **Voigt profile** (convolution of Lorentzian and Gaussian) to extract the full width at half maximum (FWHM). This is **one run** → we get one extracted linewidth $w_i$.

5. **Repeat** — Steps 1–4 are repeated $N$ times (typically $N = 2000$), producing a histogram of extracted linewidths $\{w_1, w_2, ..., w_N\}$. This is **one simulation**.

6. **Compare to experiment** — The simulated histogram is compared to the experimental linewidth distribution using a $\chi^2$ test:

$$S(\gamma, \bar{n}) = \sum_{i=0}^{N} \frac{[O_i - E_i(\gamma, \bar{n})]^2}{E_i(\gamma, \bar{n})}$$

   where $O_i$ are the observed (experimental) counts and $E_i$ the expected (simulated) counts per bin.

7. **Grid search** — The whole simulation (steps 1–6) is repeated for many combinations of $(\gamma, \bar{n})$ on a grid. The pair that minimizes $S(\gamma, \bar{n})$ is the best estimate, and the $\chi^2$ surface gives confidence intervals.

---

### Goal: Make This Differentiable

The original approach uses a brute-force **grid search** over $(\gamma, \bar{n})$. This is:
- Computationally expensive (many simulations needed)
- Limited to low-dimensional parameter sweeps
- Discrete — no gradient information

Our goal is to convert the entire pipeline into a **differentiable program** so we can optimize $(\gamma, \bar{n})$ via gradient descent instead of grid search. This introduces **two main challenges**:

**Challenge 1 — The sampling part:** Steps 1 and 3 involve random sampling (Normal and Poisson). Random sampling is not differentiable. We need a way to pass gradients through the sampling step (e.g., reparameterization trick, Gumbel-Softmax, straight-through estimator).

**Challenge 2 — The fitting part:** Step 4 involves fitting a Voigt profile to each individual spectrum. This is an iterative optimization routine (e.g., Levenberg-Marquardt) that does not backpropagate. We need a differentiable replacement for fitting — either by making the fitter itself differentiable, or by replacing it with a learned/surrogate model.

We will explore both challenges in detail in the following chapters.

---

## Chapter 2: Making the Sampling Differentiable

In the MC algorithm, the first step is to sample a photon count $n$ from a normal distribution $\mathcal{N}(\bar{n}, \sigma)$. This is straightforward in a classical simulation:

$$n \sim \mathcal{N}(\bar{n}, \sigma)$$

However, $n$ must be an integer (you can't detect half a photon), so we round:

$$n = \text{round}(\bar{n} + \sigma \cdot \varepsilon), \quad \varepsilon \sim \mathcal{N}(0,1)$$

This is where differentiability breaks down. Both the random sampling and the rounding are non-differentiable operations — they have no gradient with respect to the parameters $(\bar{n}, \sigma)$.

If we want to optimize $(\bar{n}, \sigma)$ via gradient descent, we need gradients of the loss with respect to these parameters. The random sampling blocks the gradient because it's a stochastic node. The rounding blocks it because it's a step function (gradient is zero almost everywhere).

In this chapter, we will build up the tools to solve this, starting with a **simple example** where the underlying process is clear. We will show how to:

1. Make the random sampling differentiable using the **reparameterization trick**
2. Handle the rounding using **Straight-Through Estimator (STE)** or similar approximations
3. Verify that gradients flow correctly through the pipeline

Once we understand these building blocks, we will apply them to the full MC simulation in the next chapters.

---

### Conclusion: The Two Strategies

#### 1. Reparameterization Trick

Instead of sampling $n \sim \mathcal{N}(\mu, \sigma)$ directly (a stochastic node with no gradient), we separate the randomness:

$$\varepsilon \sim \mathcal{N}(0, 1), \quad n = \mu + \sigma \cdot \varepsilon \quad \Longrightarrow \quad \frac{dn}{d\mu} = 1$$

The randomness is isolated in $\varepsilon$, which we keep fixed during the gradient step. The gradient path becomes $dL/d\mu = dL/dn \cdot dn/d\mu = dL/dn$.

#### 2. Finite Difference Surrogate Gradient

The discrete for-loop (or any non-differentiable operation) blocks autograd. We estimate the gradient by probing the loss at neighboring points:

$$\frac{dL}{dn} \approx \frac{L(n+1) - L(n-1)}{2}$$

This is a **finite difference surrogate gradient** — "finite difference" because we use numerical differences, and "surrogate" because it replaces a true gradient that doesn't exist. As long as we can evaluate the loss function, we can probe it at $n \pm 1$ and get a gradient estimate, regardless of how complex or non-differentiable the forward pass is.

**The complete pipeline:**

```
μ → reparameterization → n → discrete for-loop → x → loss
    (differentiable)           (not differentiable)
                                      ↓
                         finite difference surrogate gradient
                         probes L(n+1) and L(n-1)
```

This approach is general: it works for any non-differentiable forward pass, including the full MC simulation in the paper.

---

## Chapter 3: Differentiating Through the Fit

In the MC algorithm, after sampling photons and binning them into a spectrum, we **fit a profile** (Voigt in the paper) to extract the linewidth $w_i$. This fit is an iterative optimization — it doesn't backpropagate.

Here we tackle **Challenge 2** (independent from Challenge 1, for now) using **implicit differentiation**. The idea is simple: even though the fit is an iterative optimizer, at the *optimal* parameters $\theta^*$ the gradient of the inner loss is zero:

$$\left. \frac{\partial L_{\text{inner}}}{\partial \theta} \right|_{\theta = \theta^*} = 0$$

This **optimality condition** constrains how $\theta^*$ shifts when we change an outer parameter $\gamma_{\text{true}}$ (the linewidth of the data we generate). Differentiating it:

$$\frac{d}{d\gamma}\left(\frac{\partial L_{\text{inner}}}{\partial \theta}\right) = 0$$

$$\frac{\partial^2 L_{\text{inner}}}{\partial \theta^2} \cdot \frac{d\theta^*}{d\gamma} + \frac{\partial^2 L_{\text{inner}}}{\partial \theta \partial \gamma} = 0$$

$$\frac{d\theta^*}{d\gamma} = - \mathbf{H}^{-1} \cdot \frac{\partial^2 L_{\text{inner}}}{\partial \theta \partial \gamma}$$

We never backprop through the fit iterations — we just need the Hessian $\mathbf{H}$ and the mixed derivative at the optimum.

### Simple Example: Lorentzian Fit

Instead of a full Voigt (5 params), we start with a **Lorentzian** fit (4 params: center, $\gamma$, amplitude, offset) — the same underlying model from the paper. The pipeline:

1. Generate a noisy Lorentzian spectrum with a known linewidth $\gamma_{\text{true}}$
2. Fit a Lorentzian $\to$ extract $\gamma_{\text{fit}}$
3. Compute $\text{FWHM} = 2 \gamma_{\text{fit}}$
4. Loss $L = (\text{FWHM} - 50)^2$
5. Use **implicit differentiation** to get $dL/d\gamma_{\text{true}}$
6. Gradient descent on $\gamma_{\text{true}}$

---

### Conclusion: Implicit Differentiation

Key advantage: the fit can be solved by **any** optimizer (L-BFGS, Nelder-Mead, even manual grid search). The gradient through the fit only depends on the final optimum, not on the path taken to get there.

Key requirement: the inner loss must be twice differentiable at $\theta^*$, so we need a smooth model (Lorentzian, Voigt, etc.) and a smooth loss ($\chi^2$, log-likelihood).

This is completely orthogonal to the sampling problem (Chapter 2). Later we will combine both.

---

## Changelog

| Date | Chapter | What changed |
|------|---------|-------------|
| 14.06.2026 | — | Notebook created |
| 14.06.2026 | Terminology | Added detailed run/simulation reference |
| 14.06.2026 | Chapter 2 | Conclusion + code: reparameterization + finite difference surrogate gradient |
| 14.06.2026 | Chapter 3 | Implicit differentiation through Lorentzian fit (text + code) |
